In [16]:
import pandas as pd
import numpy as np
import pickle
import os
from scipy.ndimage import gaussian_filter1d

from IPython.display import display, HTML

# This line is for notebook display settings, not a standard library import
display(HTML("<style>:root { --jp-notebook-max-width: 90% !important; }</style>"))

In [17]:
# ==============================================================================
# --- 1. Create Penalty  ---
# ==============================================================================
def calculate_penalty_from_pkl(d_arr, param_dict, ref_dist=35000):
    """
    Vectorized penalty calculation. 
    P(d) = 1.0 for d < 35kb. Piecewise decay for d >= 35kb.
    """
    d = np.atleast_1d(d_arr).astype(float)
    results = np.ones_like(d) 
    
    mask_active = d >= ref_dist
    d_act = d[mask_active]
    
    if len(d_act) > 0:
        alphas = param_dict['alphas']
        transitions = param_dict.get('transitions_bp', [])
        bounds = [0] + transitions + [np.inf]
        
        # Internal result array for active indices to prevent dimension mismatch
        res_act = np.zeros_like(d_act)
        for i in range(len(alphas)):
            m = (d_act > bounds[i]) & (d_act <= bounds[i+1])
            if np.any(m):
                res_act[m] = d_act[m]**(-alphas[i])
        
        # Normalize: P(35kb) = 1.0
        norm_factor = ref_dist**(-alphas[0])
        results[mask_active] = res_act / norm_factor
        
    return results

# ==============================================================================
# --- 2. Configuration & Directory Setup ---
# ==============================================================================
pkl_path = "penalty_parameters.pkl"
data_dir = "/jlab/home/luca/long_ms_plots/data/"
output_dir = "../scATAC_penalized/"  # Up one level as requested

os.makedirs(output_dir, exist_ok=True)

with open(pkl_path, 'rb') as f:
    FINAL_PARAMS = pickle.load(f)

coacr_files = {
    "Soybean": f"{data_dir}cleaned_coacr_soybean.csv",
    "Rice":    f"{data_dir}cleaned_coacr_rice.csv",
    "Maize":   f"{data_dir}cleaned_coacr_maize.csv"
}

# Toggle for Global vs Tissue
USE_GLOBAL_CONSENSUS = True 
mode_str = "GLOBAL CONSENSUS" if USE_GLOBAL_CONSENSUS else "TISSUE-SPECIFIC"

In [18]:


# ==============================================================================
# --- 3. Main Application & Export Loop ---
# ==============================================================================
corrected_coacr = {}
print(f"⚙️ Starting pipeline (Mode: {mode_str})...")

for species in ["Soybean", "Rice", "Maize"]:
    try:
        # Load
        df = pd.read_csv(coacr_files[species])
        
        # Select Params
        params = FINAL_PARAMS['consensus'][species.lower()]
        
        # Apply Penalty
        df['penalty'] = calculate_penalty_from_pkl(df['distance'], params)
        df['coaccess_penalized'] = df['coaccess'] * df['penalty']
        
        # Export
        save_name = f"cleaned_coacr_{species.lower()}_penalized.csv"
        export_path = os.path.join(output_dir, save_name)
        
        # Save original columns + penalized score (dropping the helper penalty col)
        df.drop(columns=['penalty']).to_csv(export_path, index=False)
        
        corrected_coacr[species] = df
        print(f"✅ {species:8} | Exported to: {output_dir}{save_name}")
        
    except Exception as e:
        print(f"❌ {species:8} | Error: {e}")

# ==============================================================================
# --- 4. Targeted Sample Verification  ---
# ==============================================================================
print("\n" + "="*75)
print(f"📊 {mode_str} VERIFICATION (One random sample per window)")
print("="*75)

icons = {"Soybean": "🌱", "Rice": "🌾", "Maize": "🌽"}
windows = [
    ("Neutral Zone", 0, 35000), 
    ("Short-Range", 35000, 100000), 
    ("Long-Range", 100000, 1000000)
]

for species in ["Soybean", "Rice", "Maize"]:
    if species not in corrected_coacr: continue
    print(f"\n{icons[species]} {species.upper()}")
    df_spec = corrected_coacr[species]
    samples = []
    
    for label, low, high in windows:
        mask = (df_spec['distance'] >= low) & (df_spec['distance'] < high) & (df_spec['coaccess'] > 0)
        subset = df_spec[mask]
        if not subset.empty:
            s = subset.sample(1).copy()
            s['Window'] = label
            samples.append(s)
    
    if samples:
        display(pd.concat(samples)[['Window', 'distance', 'coaccess', 'penalty', 'coaccess_penalized']])

print("\n" + "-"*75)
print("GUIDE TO INTERPRETING RESULTS:")
print("1. 'penalty' = 1.0 : No weight reduction (Neutral Zone).")
print("2. SMALLER penalty values mean HEAVIER distance-based reduction.")
print("-" * 75)

⚙️ Starting pipeline (Mode: GLOBAL CONSENSUS)...
✅ Soybean  | Exported to: ../scATAC_penalized/cleaned_coacr_soybean_penalized.csv
✅ Rice     | Exported to: ../scATAC_penalized/cleaned_coacr_rice_penalized.csv
✅ Maize    | Exported to: ../scATAC_penalized/cleaned_coacr_maize_penalized.csv

📊 GLOBAL CONSENSUS VERIFICATION (One random sample per window)

🌱 SOYBEAN


,Window,distance,coaccess,penalty,coaccess_penalized
748694,Neutral Zone,5006,0.030763,1.000000e+00,3.076300e-02
5459912,Short-Range,95520,0.011791,4.987135e-02,5.880117e-04
8973090,Long-Range,318897,0.018642,2.254432e+15,4.202809e+13



🌾 RICE


,Window,distance,coaccess,penalty,coaccess_penalized
336554,Neutral Zone,30190,0.086535,1.000000,0.086535
2432958,Short-Range,77095,0.079192,0.129577,0.010262
4415335,Long-Range,102893,0.159726,0.061395,0.009806



🌽 MAIZE


,Window,distance,coaccess,penalty,coaccess_penalized
4052531,Neutral Zone,3971,0.027466,1.000000,0.027466
4501598,Short-Range,72838,0.018114,0.606500,0.010986
1184383,Long-Range,287314,0.392514,0.237783,0.093333



---------------------------------------------------------------------------
GUIDE TO INTERPRETING RESULTS:
1. 'penalty' = 1.0 : No weight reduction (Neutral Zone).
2. SMALLER penalty values mean HEAVIER distance-based reduction.
---------------------------------------------------------------------------
